# Process reviews JSONL

Reads raw gzipped JSONL review files, filters by date range / sample size / fields, and writes a clean intermediate CSV for `prepare-dataset.ipynb`.

In [1]:
import json
import gzip
from pathlib import Path
from typing import Any
import pandas as pd

In [2]:
# --- Config ---
CATEGORIES = ["Beauty_and_Personal_Care", "Clothing_Shoes_and_Jewelry"]
START_DATE: str | None = "2021-01-01"
END_DATE: str | None = "2022-12-31"
MIN_RATING: int | None = None
FIELDS_TO_KEEP: list[str] = ["user_id", "parent_asin", "rating", "timestamp"]
DATA_DIR: str = "../data"
OUTPUT_FILE: str = "reviews.csv"

## Common utils

In [3]:
def stream_jsonl(path: str, fields: list[str] | None = None):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for _, line in enumerate(f):
            obj = json.loads(line)
            if fields is not None:
                obj = {k: obj.get(k) for k in fields}
            yield obj

def _date_to_ms(date_str: str | None) -> int | None:
    if date_str is None:
        return None
    return int(pd.Timestamp(date_str, tz="UTC").timestamp() * 1000)

def load_reviews(
    categories: list[str],
    start_date: str | None = None, end_date: str | None = None,
    min_rating: int | None = None, fields: list[str] | None = None,
) -> list[dict[str, Any]]:
    start_ts = _date_to_ms(start_date)
    end_ts = _date_to_ms(end_date)
    print(f"Filtering reviews with criteria: start_date={start_date}, end_date={end_date}, min_rating={min_rating}")

    reviews: list[dict[str, Any]] = []
    for cat in categories:
        cat_reviews: list[dict[str, Any]] = []

        path = f"{DATA_DIR}/{cat}.jsonl.gz"
        print(f"Loading reviews: {path}")
        for obj in stream_jsonl(path, fields=fields):
            ts: Any = obj.get("timestamp")
            rating: Any = obj.get("rating")
            if start_ts is not None and (ts is not None and ts < start_ts):
                continue
            if end_ts is not None and (ts is not None and ts > end_ts):
                continue
            if min_rating is not None and (rating is not None and rating < min_rating):
                continue
            obj["category"] = cat
            cat_reviews.append(obj)

        reviews.extend(cat_reviews)
    return reviews

## Load and filter reviews

In [5]:
reviews = load_reviews(
    CATEGORIES, start_date=START_DATE, end_date=END_DATE,
    min_rating=MIN_RATING, fields=FIELDS_TO_KEEP,
)
print(f"Loaded {len(reviews)} reviews")

Filtering reviews with criteria: start_date=2021-01-01, end_date=2022-12-31, min_rating=None
Loading reviews: ../data/Beauty_and_Personal_Care.jsonl.gz
Loading reviews: ../data/Clothing_Shoes_and_Jewelry.jsonl.gz
Loaded 27220447 reviews


In [6]:
df_reviews = pd.DataFrame(reviews)
display(df_reviews.sample(5))
df_reviews.info()

,user_id,parent_asin,rating,timestamp,category
12651437,AHSCH6WIEPXRXF3O3VIZJZE75QVQ,B0B7CHB5YN,5.0,1667583652297,Clothing_Shoes_and_Jewelry
13894236,AEG3HPMI3IL7JAKHGTDEDBEPHVZQ,B09C6PN6FK,1.0,1630943098518,Clothing_Shoes_and_Jewelry
18843124,AHDJUL46H5V4FYF3DN4Q3JXDDB6Q,B095WD7NVK,5.0,1649357968692,Clothing_Shoes_and_Jewelry
2955526,AF5HS3ZFS2D6DZMGQ6F37N45HZFA,B0BHY9Q496,5.0,1659489004496,Beauty_and_Personal_Care
18101875,AH5VIPLIIRMYN727C6VD6HUNVMYQ,B081CNC1CN,5.0,1611629665080,Clothing_Shoes_and_Jewelry


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27220447 entries, 0 to 27220446
Data columns (total 5 columns):
 #   Column       Dtype  
---  ------       -----  
 0   user_id      object 
 1   parent_asin  object 
 2   rating       float64
 3   timestamp    int64  
 4   category     object 
dtypes: float64(1), int64(1), object(3)
memory usage: 1.0+ GB


## Deduplicate reviews

In [7]:
# Keep only the most recent review for each (user_id, parent_asin) pair
before = len(df_reviews)
df_reviews = df_reviews.sort_values("timestamp").drop_duplicates(
    subset=["user_id", "parent_asin"], keep="last"
)
df_reviews = df_reviews.reset_index(drop=True)
after = len(df_reviews)
print(f"Removed {before - after:,} duplicate (user_id, parent_asin) reviews ({after:,} unique pairs remain)")

Removed 343,836 duplicate (user_id, parent_asin) reviews (26,876,611 unique pairs remain)


## Export to CSV

In [8]:
output_path = Path(DATA_DIR)
output_path.mkdir(parents=True, exist_ok=True)
file_path = output_path / OUTPUT_FILE
df_reviews.to_csv(file_path, index=False)
print(f"Wrote {file_path} ({df_reviews.shape[0]:,} rows, {df_reviews.shape[1]} columns)")

Wrote ../data/reviews.csv (26,876,611 rows, 5 columns)
